# 1. Quickstart: the Gym-like env / agent API

This notebook shows the environment/algorithm split added in `edp/env.py` and `edp/agents.py`. The same `PageCompositionEnv` and `run_episode` runner drive three very different policies — a deterministic GAM (EDP-static), an online contextual bandit (LinTS), and the LLM-prior + SGD fusion (Bayesian-EDP) — with no policy-specific loop code.

In [ ]:
import os, sys
# notebooks live in notebooks/; make the repo root importable
sys.path.insert(0, os.path.abspath('..'))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from edp.env import PageCompositionEnv
from edp.agents import EDPAgent, BanditAgent, BayesianEDPAgent, run_episode
from edp.policies.edp import EDPPolicy
from edp.policies.bandit import BanditPolicy
from edp.policies.bayesian_edp import BayesianEDPPolicy

## The environment

`PageCompositionEnv` owns the session stream and the production reward stack: page-level attribution, multi-day delay (`delay=500` sessions), and observation noise (`noise_sigma=0.20`). Each observation exposes the fashion `category` and the 14 raw behavioural signals in `feat`; the persona is hidden because it is the latent the reward depends on.

In [ ]:
env = PageCompositionEnv(n=2000, seed=42, source='parametric',
                         delay=500, noise_sigma=0.20)
obs = env.reset()
print('category :', obs.category)
print('feat keys:', sorted(obs.feat)[:6], '...')
print('index    :', obs.index)

## One episode, three policies

The runner is identical for every agent. An agent only has to implement `act(obs) -> (page, payload)`, `learn(reward, payload)`, and `maybe_checkpoint(index)`.

In [ ]:
def fresh():
    return PageCompositionEnv(n=2000, seed=42, source='parametric',
                              delay=500, noise_sigma=0.20)

runs = {}
e = fresh(); runs['EDP-static']   = (run_episode(e, EDPAgent(EDPPolicy())), e)
e = fresh(); runs['LinTS-warm']   = (run_episode(e,
        BanditAgent(BanditPolicy(ctx_dim=7, alpha=0.3, seed=7), context='warm')), e)
e = fresh(); runs['Bayesian-EDP'] = (run_episode(e,
        BayesianEDPAgent(BayesianEDPPolicy(lr=5e-4, lam=2.0, prior_sigma=0.3))), e)

for name, (out, e) in runs.items():
    print(f'{name:14s} regret% = {out["regret_pct"]:.2f}')

## Cumulative regret over the episode

The environment records the oracle and the realised reward per session, so the cumulative-regret curve is just `cumsum(oracle - reward)`. EDP-static and Bayesian-EDP track the oracle closely from session 0 (the LLM-authored prior is competent with zero data); the bandit pays an exploration tax that never amortises under page-level reward.

In [ ]:
plt.figure(figsize=(8, 5))
for name, (out, e) in runs.items():
    cum = np.cumsum(e.oracles - e.rewards)
    plt.plot(cum, label=name, lw=2)
plt.xlabel('session'); plt.ylabel('cumulative regret')
plt.title('Cumulative regret (parametric, production conditions)')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## Switching the simulator

Pass `source='llm'` to swap the parametric 8-persona simulator for the LLM-driven 14-persona × 6-category one. Nothing else changes.

In [ ]:
e = PageCompositionEnv(n=2000, seed=42, source='llm',
                       delay=500, noise_sigma=0.20)
out = run_episode(e, EDPAgent(EDPPolicy()))
print('EDP-static on LLM-persona, regret% =', round(out['regret_pct'], 2))